# CreditLab market-data pull — run inside LSEG Workspace **Codebook**

Pulls four datasets and writes one JSON export for the local CreditLab XVA engine
(`creditlab.xva.lseg.load_lseg_export`):

1. **USD SOFR zero curve** (IPA ZC curve, with an OIS-chain fallback)
2. **Henry Hub (NYMEX NG) futures strip** — settlement prices
3. **Gas vol** — realized from the front-month continuation (override with implied if you have it)
4. **Single-name CDS spread curves** for the tickers you list — discovered via search, senior USD

**How to use:** run cells top to bottom, eyeball each cell's printout, then download the
written JSON (File → Download) and drop it into the repo's `data/processed/` directory.

**Licensing:** university LSEG access covers personal academic research. Keep the export in
`data/processed/` (gitignored) — never commit or redistribute it.

Cells are defensive: entitlements differ per account, so each pull prints what it got and
the assembly cell tells you what is still missing.

In [ ]:
import json
import math
from datetime import date

import lseg.data as ld

ld.open_session()  # Codebook's default session — no config needed here
print("session open")

In [ ]:
# ---- parameters ------------------------------------------------------------
TICKERS = ["OXY", "DVN", "APA", "HAL", "SLB", "KMI", "WMB"]  # names to try for CDS
CDS_TENORS = ["1Y", "2Y", "3Y", "5Y", "7Y", "10Y"]
N_FUTURES = 36                    # NG contracts to keep
IMPLIED_VOL_OVERRIDE = None       # e.g. 0.58 if you read ATM IV off the vol surface app
ASOF = date.today()
OUT_PATH = f"lseg_export_{ASOF.strftime('%Y%m%d')}.json"

TENOR_YEARS = {"6M": 0.5, "1Y": 1.0, "2Y": 2.0, "3Y": 3.0, "4Y": 4.0,
               "5Y": 5.0, "7Y": 7.0, "10Y": 10.0, "15Y": 15.0, "20Y": 20.0, "30Y": 30.0}
export = {"version": 1, "asof": ASOF.isoformat(),
          "source": "LSEG Workspace Codebook", "notes": []}
print(f"as-of {ASOF} → {OUT_PATH}")

In [ ]:
# ---- 1. USD SOFR zero curve ------------------------------------------------
zeros = []
try:
    from lseg.data.content.ipa.curves import zc_curves
    resp = zc_curves.Definition(
        curve_definition=zc_curves.ZcCurveDefinitions(
            currency="USD", index_name="SOFR", name="USD SOFR Swap ZC Curve"),
        curve_parameters=zc_curves.ZcCurveParameters(valuation_date=ASOF.isoformat()),
    ).get_data()
    df = resp.data.df
    # keep standard pillars; columns are endDate/tenor/discountFactor/zeroRate-ish
    for _, row in df.iterrows():
        t = TENOR_YEARS.get(str(row.get("tenor", "")))
        rate = row.get("ratePercent")
        if t and rate is not None:
            zeros.append([t, float(rate) / 100.0])
    print(f"IPA ZC curve: {len(zeros)} pillars")
except Exception as e:
    print(f"IPA curve failed ({type(e).__name__}: {e}) — falling back to OIS chain")
    try:
        ois = ld.get_data("0#USDSROIS=", ["PRIMACT_1", "MATUR_DATE"])
        for _, row in ois.dropna().iterrows():
            mat = row["MATUR_DATE"]
            t = (mat.date() - ASOF).days / 365.0 if hasattr(mat, "date") else None
            if t and 0.2 <= t <= 30:
                zeros.append([round(t, 2), float(row["PRIMACT_1"]) / 100.0])
        export["notes"].append("zeros: OIS par rates used as zeros (fallback)")
        print(f"OIS chain fallback: {len(zeros)} pillars")
    except Exception as e2:
        print(f"OIS fallback also failed: {e2}")

zeros = sorted(zeros)[:12]
export["zeros"] = zeros
zeros

In [ ]:
# ---- 2. Henry Hub futures strip ---------------------------------------------
gas = {}
try:
    fut = ld.get_data("0#NG:", ["SETTLE", "TRDPRC_1", "EXPIR_DATE"])
    fut = fut.dropna(subset=["EXPIR_DATE"]).head(N_FUTURES)
    forwards = []
    for _, row in fut.iterrows():
        px = row["SETTLE"] if row["SETTLE"] == row["SETTLE"] else row["TRDPRC_1"]
        exp = row["EXPIR_DATE"]
        d = exp.date() if hasattr(exp, "date") else date.fromisoformat(str(exp)[:10])
        if px == px and d > ASOF:
            forwards.append([d.isoformat(), float(px)])
    forwards.sort()
    gas["forwards"] = forwards
    gas["spot"] = forwards[0][1] if forwards else None
    print(f"NG strip: {len(forwards)} contracts, front {gas['spot']}")
except Exception as e:
    print(f"NG chain failed: {type(e).__name__}: {e}")
gas

In [ ]:
# ---- 3. gas vol -------------------------------------------------------------
if IMPLIED_VOL_OVERRIDE:
    gas["implied_vol"] = float(IMPLIED_VOL_OVERRIDE)
    export["notes"].append("vol: manual implied-vol override")
else:
    try:
        hist = ld.get_history("NGc1", fields="TRDPRC_1", interval="daily", count=260)
        px = hist.dropna().iloc[:, 0].astype(float)
        rets = [math.log(b / a) for a, b in zip(px, px[1:]) if a > 0 and b > 0]
        mean = sum(rets) / len(rets)
        sig = math.sqrt(sum((r - mean) ** 2 for r in rets) / (len(rets) - 1)) * math.sqrt(252)
        gas["implied_vol"] = round(sig, 4)
        export["notes"].append("vol: realized from NGc1 (set IMPLIED_VOL_OVERRIDE for true implied)")
        print(f"realized vol (NGc1, 1y): {sig:.1%}")
    except Exception as e:
        print(f"vol pull failed: {e} — set IMPLIED_VOL_OVERRIDE and rerun")
export["gas"] = gas

In [ ]:
# ---- 4. single-name CDS curves ----------------------------------------------
# Discover each name's senior-USD 5Y CDS RIC via search, then substitute tenors.
cds = {}
for tk in TICKERS:
    try:
        hits = ld.discovery.search(
            query=f"{tk} credit default swap senior USD 5Y",
            select="RIC,DocumentTitle", top=10,
        )
        ric5 = None
        for _, h in hits.iterrows():
            ric = str(h.get("RIC", ""))
            if "5Y" in ric and ric.endswith("=R"):
                ric5 = ric
                break
        if not ric5:
            print(f"{tk}: no senior-USD 5Y CDS RIC found — skipped")
            continue
        rics = {t: ric5.replace("5Y", t) for t in CDS_TENORS}
        quotes = ld.get_data(list(rics.values()), ["PRIMACT_1", "MID_SPREAD"])
        spreads = []
        for t, ric in rics.items():
            row = quotes[quotes["Instrument"] == ric]
            if row.empty:
                continue
            val = row.iloc[0].get("MID_SPREAD")
            if val != val or val is None:
                val = row.iloc[0].get("PRIMACT_1")
            if val == val and val is not None:
                spreads.append([TENOR_YEARS[t], float(val) / 10_000.0])  # bp → decimal
        if spreads:
            cds[tk] = {"recovery": 0.4, "spreads": sorted(spreads)}
            print(f"{tk}: {len(spreads)} tenors from {ric5}")
        else:
            print(f"{tk}: RIC family found but no quotes — check entitlements")
    except Exception as e:
        print(f"{tk}: failed ({type(e).__name__}: {e})")
export["cds"] = cds
print(f"\nCDS curves: {sorted(cds)}")

In [ ]:
# ---- assemble, validate, write ----------------------------------------------
problems = []
if not export.get("zeros"):
    problems.append("zeros missing")
if not export.get("gas", {}).get("forwards"):
    problems.append("NG forwards missing")
if not export.get("gas", {}).get("implied_vol"):
    problems.append("vol missing")
if not export.get("cds"):
    problems.append("no CDS curves (spread-based CVA won't be available)")

if problems:
    print("INCOMPLETE — fix before using locally:", "; ".join(problems))
else:
    print("export complete")

with open(OUT_PATH, "w") as f:
    json.dump(export, f, indent=1)
print(f"wrote {OUT_PATH} — download it (File → Download) into data/processed/")